In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from datasets import load_dataset
dataset = load_dataset("dp1812/celestial-spiritual-conversations-v2", split = 'train')
print(dataset)

Dataset({
    features: ['messages', 'category', 'conversation_type', 'tone'],
    num_rows: 2700
})


In [ ]:
dataset[0]

{'messages': [{'content': "I'm struggling with meditation. Can you help?",
   'role': 'user'},
  {'content': "🧘\u200d♀️ Meditation struggles are completely normal! Start small - even 5 minutes daily. Focus on your breath, and when your mind wanders (it will!), gently return to breathing. Don't judge yourself; each moment of awareness is progress. Consider guided meditations or mantra repetition to anchor your practice.",
   'role': 'assistant'}],
 'category': 'spiritual_conversation',
 'conversation_type': 'meditation_help',
 'tone': 'warm_spiritual'}

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("---DOWNLOADING LLAMA-3.1-8B-INSTRUCT (4-BIT)---")
model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,  #for training not generation. Training example should not exceed this length
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("ATTACHING LORA ADAPTERS")
model = FastLanguageModel.get_peft_model(
    model,
    r= 16, #rank of adapter. 16 is sweet spot
    lora_alpha=16,
    lora_dropout=0, #best for unsloth
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj","up_proj",'down_proj'],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state = 3407,
    use_rslora=False,
    loftq_config = None
)

print(f"Successfully loaded.")
model.print_trainable_parameters()

---DOWNLOADING LLAMA-3.1-8B-INSTRUCT (4-BIT)---
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:53: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


ATTACHING LORA ADAPTERS
Successfully loaded.
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
from unsloth.chat_templates import get_chat_template

#get Llama3 's exact chat template into our tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1"
)

def formatting_prompts_func(examples):
  convos = examples['messages']

  texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
  return {"text":texts,}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset['text'][0])

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm struggling with meditation. Can you help?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

🧘‍♀️ Meditation struggles are completely normal! Start small - even 5 minutes daily. Focus on your breath, and when your mind wanders (it will!), gently return to breathing. Don't judge yourself; each moment of awareness is progress. Consider guided meditations or mantra repetition to anchor your practice.<|eot_id|>


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",  #column we created
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    dataset_num_proc =2,
    packing=True,
    args = TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        weight_decay=0.01,
        seed = 3407,
    ),
)
print("--- IGNITION: STARTING TRAINING ---")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2700 [00:00<?, ? examples/s]

--- IGNITION: STARTING TRAINING ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.710068
2,3.780880
3,3.473395
4,3.070748
5,2.642947
6,2.438442
7,1.957746
8,1.666620
9,1.406078
10,1.259104


In [ ]:
print("--- MERGING LORA AND EXPORTING TO GGUF (4-BIT) ---")

# This will take 5-10 minutes. It is merging the weights and quantizing them.
model.save_pretrained_gguf(
    "gita_spiritual_model",
    tokenizer,
    quantization_method = "q4_k_m"
)

print("--- EXPORT COMPLETE ---")

--- MERGING LORA AND EXPORTING TO GGUF (4-BIT) ---
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [12:47<00:00, 191.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:49<00:00, 117.47s/it]


Unsloth: Merge process complete. Saved to `/content/gita_spiritual_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gita_spiritual_model_gguf/Meta-Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gita_spiritual_model_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model gita_spiritual_model_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to gita_spiritual_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f gita_spiritual_model_gguf/Modelfile
--- EXPORT COMPLETE ---
